# GFP 설계 후보 4개 검증 · v0.1.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/diagnostics/GFP_Design_Validation_T4.ipynb?validation_version=0.1.0)

Run 02의 설계 후보 4개를 도킹하고 기존 RF3와 목표 접촉 개수를 비교합니다. T4 GPU에서 전체 실행하면 결과 ZIP 하나를 다운로드합니다.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
from collections import deque
from IPython.display import display
import json
import pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython==1.85"], check=True)
log = checkout.parent / "candidate_validation.log"
with log.open("w") as handle:
    execution = subprocess.run(
        [sys.executable, str(checkout / "gfp_pipeline/candidate_validation.py"), revision],
        stdout=handle, stderr=subprocess.STDOUT,
    )
with log.open() as handle:
    tail = deque(handle, maxlen=40)
if execution.returncode:
    print("".join(tail))
    print(log)
    execution.check_returncode()
result_zip = Path(tail[-1].strip())
if not result_zip.is_file():
    raise FileNotFoundError(result_zip)
display(pd.DataFrame(json.loads((result_zip.with_suffix("") / "candidate_summary.json").read_bytes())))


In [ ]:
from google.colab import files
files.download(str(result_zip))
